FinalV2

In [4]:
"""
PSIms Complete Unified System v2.0
Integrated file upload, validation, CSV conversion, and analytics with Tkinter UI

Author: Subhoit Talukdar
Version: 2.0 - Complete Integration
"""

import pandas as pd
import numpy as np
import os
import json
import tkinter as tk
from tkinter import filedialog, messagebox, ttk, scrolledtext
from pathlib import Path
from datetime import datetime
import openpyxl
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')


# =====================================================
# CONFIGURATION MANAGER
# =====================================================

class PSImsConfig:
    """Manages system configuration"""
    
    def __init__(self):
        self.config_file = 'psims_config.json'
        self.config = self.load_or_create()
    
    def load_or_create(self):
        """Load config or create with default structure"""
        default_config = {
            'folders': {
                'input_folder': '',
                'csv_output': '',
                'results_output': ''
            },
            'last_run': None
        }
        
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, 'r') as f:
                    loaded_config = json.load(f)
                    
                    # Ensure the loaded config has the required structure
                    if 'folders' not in loaded_config:
                        loaded_config['folders'] = default_config['folders']
                    else:
                        # Ensure all required folder keys exist
                        for key in default_config['folders']:
                            if key not in loaded_config['folders']:
                                loaded_config['folders'][key] = ''
                    
                    return loaded_config
            except (json.JSONDecodeError, KeyError, Exception) as e:
                print(f"Warning: Config file corrupted, creating new one: {e}")
                return default_config
        
        return default_config
    
    def save(self):
        """Save configuration to file"""
        try:
            with open(self.config_file, 'w') as f:
                json.dump(self.config, f, indent=4)
        except Exception as e:
            print(f"Warning: Could not save config: {e}")
    
    def update_folder(self, key, path):
        """Update folder path in config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        self.config['folders'][key] = path
        self.save()
    
    def get_folder(self, key):
        """Get folder path from config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        return self.config['folders'].get(key, '')


# =====================================================
# SCORING CONFIGURATION
# =====================================================

SCORING_CONFIG = {
    'engagement': {
        'email_open_weight': 0.50, 'email_click_weight': 0.50,
        'whatsapp_read_weight': 0.50, 'whatsapp_click_weight': 0.50,
        'call_productive_weight': 0.70, 'call_duration_weight': 0.30,
        'final_email_weight': 0.33, 'final_whatsapp_weight': 0.33, 
        'final_call_weight': 0.34, 'max_score': 100
    },
    'academic': {
        'publication_points_per_item': 5, 'publication_max_score': 40,
        'trial_points_per_item': 15, 'trial_max_score': 30,
        'association_points_per_item': 10, 'association_max_score': 30,
        'max_score': 100
    },
    'social_media': {
        'follower_log_multiplier': 10, 'follower_max_score': 50, 
        'follower_min_threshold': 100,
        'platform_points_per_item': 10, 'platform_max_score': 30,
        'healthcare_platform_points_per_item': 10, 
        'healthcare_platform_max_score': 20,
        'max_score': 100
    },
    'recognition': {
        'award_points_per_item': 15, 'award_max_score': 30,
        'press_points_per_item': 10, 'press_max_score': 25,
        'event_points_per_item': 8, 'event_max_score': 25,
        'association_points_per_item': 5, 'association_max_score': 20,
        'max_score': 100
    }
}


# =====================================================
# SMART EXCEL CONVERTER
# =====================================================

class SmartConverter:
    """Intelligently combines and converts Excel files"""
    
    def __init__(self, output_folder, log_callback=None):
        self.output_folder = output_folder
        self.log = log_callback or print
        self.warnings = []
    
    def standardize_name(self, name):
        """Standardize names (lowercase, no spaces)"""
        return name.strip().lower().replace(' ', '_').replace("'", "")
    
    def fuzzy_match(self, str1, str2, threshold=0.8):
        """Fuzzy string matching"""
        return SequenceMatcher(None, str1.lower(), str2.lower()).ratio() >= threshold
    
    def combine_pi_batches(self, pi_files):
        """Combine multiple PI batch files"""
        self.log("\n📊 Combining Personal Info Batches...")
        
        # Step 1: Get all sheets from all files
        all_sheets_per_file = {}
        for file in pi_files:
            wb = openpyxl.load_workbook(file, read_only=True)
            all_sheets_per_file[file] = wb.sheetnames
            wb.close()
            self.log(f"   {os.path.basename(file)}: {len(all_sheets_per_file[file])} sheets")
        
        # Step 2: Match sheets across files
        master_sheets = all_sheets_per_file[pi_files[0]]  # Use first file as master
        sheet_mapping = {sheet: [] for sheet in master_sheets}
        
        for file in pi_files:
            file_sheets = all_sheets_per_file[file]
            for master_sheet in master_sheets:
                # Try exact match first
                if master_sheet in file_sheets:
                    sheet_mapping[master_sheet].append((file, master_sheet))
                else:
                    # Try fuzzy match
                    for file_sheet in file_sheets:
                        if self.fuzzy_match(master_sheet, file_sheet):
                            warning = f"Fuzzy match: '{master_sheet}' → '{file_sheet}' in {os.path.basename(file)}"
                            self.warnings.append(warning)
                            self.log(f"   ⚠️  {warning}")
                            sheet_mapping[master_sheet].append((file, file_sheet))
                            break
        
        # Step 3: Combine sheets
        self.log("\n   📝 Combining sheets...")
        for sheet_name, file_sheet_pairs in sheet_mapping.items():
            if not file_sheet_pairs:
                self.log(f"   ⚠️  No data found for sheet: {sheet_name}")
                continue
            
            dfs = []
            for file, actual_sheet_name in file_sheet_pairs:
                try:
                    df = pd.read_excel(file, sheet_name=actual_sheet_name)
                    
                    # Standardize column names (lowercase, no spaces)
                    df.columns = [self.standardize_name(col) for col in df.columns]
                    
                    dfs.append(df)
                    self.log(f"      ✓ {os.path.basename(file)} → {actual_sheet_name}: {len(df)} rows")
                except Exception as e:
                    self.log(f"      ❌ Error reading {actual_sheet_name} from {os.path.basename(file)}: {e}")
            
            if dfs:
                # Combine DataFrames
                combined = pd.concat(dfs, ignore_index=True)
                
                # Remove duplicates based on UIN if present
                if 'uin' in combined.columns:
                    before = len(combined)
                    combined = combined.drop_duplicates(subset=['uin'], keep='last')
                    after = len(combined)
                    if before != after:
                        self.log(f"         Removed {before - after} duplicate UINs")
                
                # Save with standardized name
                output_name = self.standardize_name(sheet_name) + '.csv'
                output_path = os.path.join(self.output_folder, output_name)
                combined.to_csv(output_path, index=False, encoding='utf-8')
                self.log(f"      ✓ Saved {output_name}: {len(combined)} rows, {len(combined.columns)} cols")
    
    def convert_engagement_files(self, engagement_files):
        """Convert engagement files with standardized names"""
        self.log("\n📅 Converting Engagement Files...")
        
        for file in engagement_files:
            try:
                df = pd.read_excel(file)
                
                # Standardize column names
                df.columns = [self.standardize_name(col) for col in df.columns]
                
                # Generate standardized filename
                original_name = os.path.basename(file).replace('.xlsx', '').replace('.xls', '')
                standardized_name = self.standardize_name(original_name) + '.csv'
                
                output_path = os.path.join(self.output_folder, standardized_name)
                df.to_csv(output_path, index=False, encoding='utf-8')
                
                self.log(f"   ✓ {original_name} → {standardized_name}: {len(df)} rows")
            except Exception as e:
                self.log(f"   ❌ Failed to convert {os.path.basename(file)}: {e}")
    
    def get_warnings(self):
        return self.warnings


# =====================================================
# PSIMS ANALYSIS ENGINE
# =====================================================

class PSImsEngine:
    """Core analytics engine"""
    
    def __init__(self, csv_folder, output_folder, log_callback=None, eligibility_mode='relaxed'):
        self.csv_folder = csv_folder
        self.output_folder = output_folder
        self.log = log_callback or print
        self.data = {}
        self.eligibility_mode = eligibility_mode
        
        # Eligibility rules
        self.eligibility_rules = {
            'strict': {'min_buckets': 4, 'engagement_required': False},
            'moderate': {'min_buckets': 3, 'engagement_required': False},
            'relaxed': {'min_buckets': 2, 'engagement_required': False},
            'basic': {'min_buckets': 1, 'engagement_required': False}
        }
    
    def load_csv(self, filename):
        """Safely load CSV"""
        filepath = os.path.join(self.csv_folder, filename)
        try:
            if os.path.exists(filepath):
                df = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
                return df
            return pd.DataFrame()
        except:
            try:
                return pd.read_csv(filepath, encoding='latin-1', low_memory=False)
            except:
                return pd.DataFrame()
    
    def standardize_uin_column(self, df, is_engagement=False):
        """Standardize UIN column name to lowercase 'uin'"""
        if df.empty:
            return df
        
        uin_variations = [
            'UIN', 'uin', 'Uin', 'uIN',
            'Client doctor code', 'client doctor code', 'CLIENT DOCTOR CODE',
            'Client_doctor_code', 'client_doctor_code',
            'ClientDoctorCode', 'clientdoctorcode',
            'Doctor Code', 'doctor code', 'DOCTOR CODE',
            'Doctor_Code', 'doctor_code',
            'DoctorCode', 'doctorcode',
            'doctor_id', 'DoctorID', 'doctor_uin',
            'Client Code', 'client code', 'CLIENT CODE',
            'Client_Code', 'client_code',
            'id', 'ID', 'Id'
        ]
        
        for col in df.columns:
            if col in uin_variations:
                df = df.rename(columns={col: 'uin'})
                if is_engagement:
                    self.log(f"      Mapped '{col}' → 'uin'")
                return df
        
        for col in df.columns:
            col_lower = col.lower()
            if any(term in col_lower for term in ['doctor', 'code', 'uin', 'client']):
                if 'code' in col_lower or 'uin' in col_lower or 'id' in col_lower:
                    df = df.rename(columns={col: 'uin'})
                    if is_engagement:
                        self.log(f"      Mapped '{col}' → 'uin' (fuzzy match)")
                    return df
        
        if is_engagement:
            self.log(f"      ⚠ Could not find UIN column")
            self.log(f"      Available: {list(df.columns)[:10]}")
        
        return df
    
    def standardize_engagement_columns(self, df):
        """Standardize engagement metric column names"""
        column_mappings = {
            'HCP Email Open Rate': 'hcp_email_open_rate',
            'HCP Email Click Rate': 'hcp_email_click_rate',
            'Email Open Rate': 'hcp_email_open_rate',
            'Email Click Rate': 'hcp_email_click_rate',
            'HCP Whatsapp Read Rate': 'hcp_whatsapp_read_rate',
            'HCP Whatsapp Click Rate': 'hcp_whatsapp_click_rate',
            'Whatsapp Read Rate': 'hcp_whatsapp_read_rate',
            'Whatsapp Click Rate': 'hcp_whatsapp_click_rate',
            'HCP Call Productive Rate': 'hcp_call_productive_rate',
            'Call Productive Rate': 'hcp_call_productive_rate',
            'Average Duration Connected Calls': 'average_duration_connected_calls',
            'Average Call Duration': 'average_duration_connected_calls',
            'Call Duration': 'average_duration_connected_calls'
        }
        
        rename_dict = {}
        for col in df.columns:
            if col in column_mappings:
                rename_dict[col] = column_mappings[col]
            else:
                for original, standardized in column_mappings.items():
                    if col.lower() == original.lower():
                        rename_dict[col] = standardized
                        break
        
        if rename_dict:
            df = df.rename(columns=rename_dict)
            self.log(f"      Standardized {len(rename_dict)} engagement columns")
        
        return df
    
    def load_all_data(self):
        """Load all CSV files with UIN standardization"""
        self.log("\n📂 Loading Data Files...")
        
        files = {
            'pi': 'pi.csv',
            'clinics': 'clinics.csv',
            'publications': 'publication.csv',
            'trials': 'trials.csv',
            'academic_associations': 'academic_association.csv',
            'digital': 'digital.csv',
            'healthcare_platforms': 'healthcare_platforms.csv',
            'awards': 'awards.csv',
            'press': 'press.csv',
            'events': 'events.csv',
            'associations': 'association.csv'
        }
        
        for key, filename in files.items():
            self.data[key] = self.load_csv(filename)
            if not self.data[key].empty:
                self.data[key] = self.standardize_uin_column(self.data[key])
                self.log(f"   ✓ {key}: {len(self.data[key])} rows")
        
        engagement_files = [f for f in os.listdir(self.csv_folder) 
                          if f.endswith('.csv') and any(month in f.lower() 
                          for month in ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])]
        
        engagement_dfs = []
        for eng_file in engagement_files:
            df = self.load_csv(eng_file)
            if not df.empty:
                self.log(f"   📋 {eng_file} columns: {list(df.columns)[:5]}...")
                df = self.standardize_uin_column(df, is_engagement=True)
                
                if 'uin' in df.columns:
                    df = self.standardize_engagement_columns(df)
                    engagement_dfs.append(df)
                    self.log(f"   ✓ {eng_file}: {len(df)} rows, {df['uin'].nunique()} unique UINs")
                else:
                    self.log(f"   ⚠ {eng_file}: No UIN column found - skipping")
        
        if engagement_dfs:
            combined = pd.concat(engagement_dfs, ignore_index=True)
            
            if 'uin' not in combined.columns:
                self.log("   ❌ No UIN column in engagement data")
                self.data['engagement'] = pd.DataFrame()
                return
            
            expected_cols = {
                'hcp_email_open_rate': 0,
                'hcp_email_click_rate': 0,
                'hcp_whatsapp_read_rate': 0,
                'hcp_whatsapp_click_rate': 0,
                'hcp_call_productive_rate': 0,
                'average_duration_connected_calls': 0
            }
            
            for col, default in expected_cols.items():
                if col not in combined.columns:
                    combined[col] = default
                else:
                    combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(default)
            
            agg_dict = {col: 'mean' for col in expected_cols.keys()}
            self.data['engagement'] = combined.groupby('uin', as_index=False).agg(agg_dict)
            self.log(f"   ✓ Engagement (averaged): {len(self.data['engagement'])} UINs")
        else:
            self.data['engagement'] = pd.DataFrame()
    
    def aggregate_by_uin(self):
        """Aggregate all data sources by UIN"""
        self.log("\n🔗 Aggregating by UIN...")
        
        if self.data['pi'].empty:
            self.log("   ❌ No PI data")
            return pd.DataFrame()
        
        master = self.data['pi'][['uin']].drop_duplicates().copy()
        self.log(f"   Master: {len(master)} UINs")
        
        metrics = ['publication_count', 'trial_count', 'academic_association_count',
                  'award_count', 'platform_count', 'total_followers',
                  'healthcare_platform_count', 'press_count', 'event_count', 
                  'association_count']
        for metric in metrics:
            master[metric] = 0
        
        if not self.data['publications'].empty and 'uin' in self.data['publications'].columns:
            pubs = self.data['publications'].groupby('uin').size().reset_index(name='publication_count')
            master = master.merge(pubs, on='uin', how='left', suffixes=('', '_new'))
            master['publication_count'] = master['publication_count_new'].fillna(0)
            master.drop(['publication_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['trials'].empty and 'uin' in self.data['trials'].columns:
            trials = self.data['trials'].groupby('uin').size().reset_index(name='trial_count')
            master = master.merge(trials, on='uin', how='left', suffixes=('', '_new'))
            master['trial_count'] = master['trial_count_new'].fillna(0)
            master.drop(['trial_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['academic_associations'].empty and 'uin' in self.data['academic_associations'].columns:
            acad = self.data['academic_associations'].groupby('uin').size().reset_index(name='academic_association_count')
            master = master.merge(acad, on='uin', how='left', suffixes=('', '_new'))
            master['academic_association_count'] = master['academic_association_count_new'].fillna(0)
            master.drop(['academic_association_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['awards'].empty and 'uin' in self.data['awards'].columns:
            awards = self.data['awards'].groupby('uin').size().reset_index(name='award_count')
            master = master.merge(awards, on='uin', how='left', suffixes=('', '_new'))
            master['award_count'] = master['award_count_new'].fillna(0)
            master.drop(['award_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['digital'].empty and 'uin' in self.data['digital'].columns:
            digital_clean = self.data['digital'][(self.data['digital']['uin'].notna()) & 
                                                 (self.data['digital']['uin'] != '')].copy()
            if len(digital_clean) > 0:
                agg_dict = {'platform_count': ('uin', 'count')}
                if 'sm_followers' in digital_clean.columns:
                    digital_clean['sm_followers'] = pd.to_numeric(digital_clean['sm_followers'], errors='coerce').fillna(0)
                    agg_dict['total_followers'] = ('sm_followers', 'sum')
                
                digital = digital_clean.groupby('uin').agg(**agg_dict).reset_index()
                master = master.merge(digital, on='uin', how='left', suffixes=('_old', ''))
                master.drop([c for c in master.columns if c.endswith('_old')], axis=1, errors='ignore', inplace=True)
                master['platform_count'] = master.get('platform_count', 0).fillna(0)
                master['total_followers'] = master.get('total_followers', 0).fillna(0)
        
        if not self.data['healthcare_platforms'].empty and 'uin' in self.data['healthcare_platforms'].columns:
            hc = self.data['healthcare_platforms'].groupby('uin').size().reset_index(name='healthcare_platform_count')
            master = master.merge(hc, on='uin', how='left', suffixes=('', '_new'))
            master['healthcare_platform_count'] = master['healthcare_platform_count_new'].fillna(0)
            master.drop(['healthcare_platform_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['press'].empty and 'uin' in self.data['press'].columns:
            press = self.data['press'].groupby('uin').size().reset_index(name='press_count')
            master = master.merge(press, on='uin', how='left', suffixes=('', '_new'))
            master['press_count'] = master['press_count_new'].fillna(0)
            master.drop(['press_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['events'].empty and 'uin' in self.data['events'].columns:
            events = self.data['events'].groupby('uin').size().reset_index(name='event_count')
            master = master.merge(events, on='uin', how='left', suffixes=('', '_new'))
            master['event_count'] = master['event_count_new'].fillna(0)
            master.drop(['event_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['associations'].empty and 'uin' in self.data['associations'].columns:
            assoc = self.data['associations'].groupby('uin').size().reset_index(name='association_count')
            master = master.merge(assoc, on='uin', how='left', suffixes=('', '_new'))
            master['association_count'] = master['association_count_new'].fillna(0)
            master.drop(['association_count_new'], axis=1, errors='ignore', inplace=True)
        
        if not self.data['engagement'].empty:
            merged = master.merge(self.data['engagement'], on='uin', how='left')
        else:
            merged = master.copy()
            for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                       'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 
                       'average_duration_connected_calls']:
                merged[col] = 0
        
        if 'full_name' in self.data['pi'].columns:
            pi_cols = ['uin'] + [c for c in ['full_name', 'mobile', 'whatsapp_phone', 
                                             'email_id_1', 'specialty'] if c in self.data['pi'].columns]
            merged = merged.merge(self.data['pi'][pi_cols].drop_duplicates('uin'), on='uin', how='left')
        
        merged.fillna(0, inplace=True)
        return merged
    
    def calculate_scores(self, df):
        """Calculate 4 bucket scores"""
        self.log("\n🎯 Calculating Scores...")
        
        def calc_engagement(row):
            email_score = (row.get('hcp_email_open_rate', 0) * SCORING_CONFIG['engagement']['email_open_weight'] +
                          row.get('hcp_email_click_rate', 0) * SCORING_CONFIG['engagement']['email_click_weight'])
            wa_score = (row.get('hcp_whatsapp_read_rate', 0) * SCORING_CONFIG['engagement']['whatsapp_read_weight'] +
                       row.get('hcp_whatsapp_click_rate', 0) * SCORING_CONFIG['engagement']['whatsapp_click_weight'])
            call_dur = row.get('average_duration_connected_calls', 0)
            call_dur_norm = min((call_dur / 60) * 100, 100) if call_dur > 0 else 0
            call_score = (row.get('hcp_call_productive_rate', 0) * SCORING_CONFIG['engagement']['call_productive_weight'] +
                         call_dur_norm * SCORING_CONFIG['engagement']['call_duration_weight'])
            final = (email_score * SCORING_CONFIG['engagement']['final_email_weight'] +
                    wa_score * SCORING_CONFIG['engagement']['final_whatsapp_weight'] +
                    call_score * SCORING_CONFIG['engagement']['final_call_weight'])
            return round(final, 2)
        
        def calc_academic(row):
            pub = min(row['publication_count'] * SCORING_CONFIG['academic']['publication_points_per_item'],
                     SCORING_CONFIG['academic']['publication_max_score'])
            trial = min(row['trial_count'] * SCORING_CONFIG['academic']['trial_points_per_item'],
                       SCORING_CONFIG['academic']['trial_max_score'])
            assoc = min(row['academic_association_count'] * SCORING_CONFIG['academic']['association_points_per_item'],
                       SCORING_CONFIG['academic']['association_max_score'])
            return round(min(pub + trial + assoc, SCORING_CONFIG['academic']['max_score']), 2)
        
        def calc_social(row):
            followers = row['total_followers']
            follower_score = min(np.log10(followers) * SCORING_CONFIG['social_media']['follower_log_multiplier'],
                               SCORING_CONFIG['social_media']['follower_max_score']) if followers >= 100 else 0
            platform = min(row['platform_count'] * SCORING_CONFIG['social_media']['platform_points_per_item'],
                          SCORING_CONFIG['social_media']['platform_max_score'])
            hc = min(row['healthcare_platform_count'] * SCORING_CONFIG['social_media']['healthcare_platform_points_per_item'],
                    SCORING_CONFIG['social_media']['healthcare_platform_max_score'])
            return round(min(follower_score + platform + hc, SCORING_CONFIG['social_media']['max_score']), 2)
        
        def calc_recognition(row):
            award = min(row['award_count'] * SCORING_CONFIG['recognition']['award_points_per_item'],
                       SCORING_CONFIG['recognition']['award_max_score'])
            press = min(row['press_count'] * SCORING_CONFIG['recognition']['press_points_per_item'],
                       SCORING_CONFIG['recognition']['press_max_score'])
            event = min(row['event_count'] * SCORING_CONFIG['recognition']['event_points_per_item'],
                       SCORING_CONFIG['recognition']['event_max_score'])
            assoc = min(row['association_count'] * SCORING_CONFIG['recognition']['association_points_per_item'],
                       SCORING_CONFIG['recognition']['association_max_score'])
            return round(min(award + press + event + assoc, SCORING_CONFIG['recognition']['max_score']), 2)
        
        df['engagement_score'] = df.apply(calc_engagement, axis=1)
        df['academic_score'] = df.apply(calc_academic, axis=1)
        df['social_media_score'] = df.apply(calc_social, axis=1)
        df['recognition_score'] = df.apply(calc_recognition, axis=1)
        
        df['engagement_data_available'] = df['engagement_score'] > 0
        df['academic_data_available'] = (df['publication_count'] + df['trial_count'] + df['academic_association_count']) > 0
        df['social_media_data_available'] = (df['platform_count'] + df['healthcare_platform_count']) > 0
        df['recognition_data_available'] = (df['award_count'] + df['press_count'] + df['event_count'] + df['association_count']) > 0
        
        df['buckets_with_data'] = (df['engagement_data_available'].astype(int) +
                                   df['academic_data_available'].astype(int) +
                                   df['social_media_data_available'].astype(int) +
                                   df['recognition_data_available'].astype(int))
        
        self.log(f"   ✓ Scored {len(df)} doctors")
        self.log(f"   Avg Engagement: {df['engagement_score'].mean():.2f}")
        self.log(f"   Avg Academic: {df['academic_score'].mean():.2f}")
        self.log(f"   Avg Social Media: {df['social_media_score'].mean():.2f}")
        self.log(f"   Avg Recognition: {df['recognition_score'].mean():.2f}")
        
        return df
    
    def perform_clustering(self, df, n_clusters=6):
        """Cluster with Cluster 0 for zero-data doctors"""
        self.log(f"\n🔍 Clustering (k={n_clusters})...")
        
        zero_data = df[df['buckets_with_data'] == 0].copy()
        has_data = df[df['buckets_with_data'] > 0].copy()
        
        self.log(f"   Zero-data doctors: {len(zero_data)}")
        self.log(f"   Doctors with data: {len(has_data)}")
        
        zero_data['cluster_id'] = 0
        zero_data['cluster_name'] = 'Cluster 0: No Data'
        
        if len(has_data) >= (n_clusters - 1):
            score_cols = ['engagement_score', 'academic_score', 'social_media_score', 'recognition_score']
            for col in score_cols:
                has_data[col] = has_data[col].fillna(0)
            
            features = has_data[score_cols].values
            features = np.nan_to_num(features, nan=0.0)
            
            scaler = StandardScaler()
            features_scaled = scaler.fit_transform(features)
            
            kmeans = KMeans(n_clusters=n_clusters-1, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(features_scaled)
            has_data['cluster_id'] = clusters + 1
            
            centroids = kmeans.cluster_centers_
            traits = ['Engagement', 'Academic', 'Social Media', 'Recognition']
            
            for i in range(n_clusters - 1):
                cluster_data = has_data[has_data['cluster_id'] == i+1]
                centroid = centroids[i]
                centroid_dict = {traits[j]: centroid[j] for j in range(4)}
                dominant = max(centroid_dict, key=centroid_dict.get)
                
                has_data.loc[has_data['cluster_id'] == i+1, 'cluster_name'] = f"Cluster {i+1}: {dominant}-Focused"
                self.log(f"   Cluster {i+1}: {len(cluster_data)} doctors - {dominant}-Focused")
        else:
            has_data['cluster_id'] = 1
            has_data['cluster_name'] = 'Cluster 1: Mixed'
        
        result = pd.concat([zero_data, has_data], ignore_index=True)
        return result
    
    def generate_output(self, df):
        """Save results"""
        self.log("\n💾 Generating Output...")
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = os.path.join(self.output_folder, f'psims_results_{timestamp}.csv')
        
        output_cols = ['uin', 'full_name', 'specialty', 'mobile', 'engagement_score', 
                      'academic_score', 'social_media_score', 'recognition_score',
                      'buckets_with_data', 'cluster_id', 'cluster_name']
        
        output_cols = [c for c in output_cols if c in df.columns]
        output_df = df[output_cols]
        
        output_df.to_csv(output_file, index=False, encoding='utf-8')
        self.log(f"   ✓ Saved: {output_file}")
        
        self.log(f"\n📊 Summary:")
        self.log(f"   Total doctors: {len(df)}")
        self.log(f"   Cluster 0 (No data): {(df['cluster_id'] == 0).sum()}")
        for i in range(1, 6):
            count = (df['cluster_id'] == i).sum()
            if count > 0:
                self.log(f"   Cluster {i}: {count}")
        
        return output_file
    
    def run_full_analysis(self):
        """Run complete pipeline"""
        try:
            self.load_all_data()
            merged = self.aggregate_by_uin()
            if merged.empty:
                return None
            
            scored = self.calculate_scores(merged)
            clustered = self.perform_clustering(scored)
            output_file = self.generate_output(clustered)
            return output_file
        except Exception as e:
            self.log(f"\n❌ Analysis failed: {e}")
            import traceback
            traceback.print_exc()
            return None


# =====================================================
# TKINTER UI
# =====================================================

class PSImsUI:
    """Main UI for PSIms system"""
    
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("PSIms - Healthcare Analytics System v2.0")
        self.root.geometry("900x700")
        self.root.resizable(True, True)
        
        self.config = PSImsConfig()
        self.pi_files = []
        self.engagement_files = []
        
        self.setup_ui()
    
    def setup_ui(self):
        """Create UI layout"""
        # Title
        title_frame = tk.Frame(self.root, bg="#2c3e50", height=80)
        title_frame.pack(fill="x")
        title_frame.pack_propagate(False)
        
        title = tk.Label(title_frame, text="PSIms Healthcare Analytics", 
                        font=("Arial", 24, "bold"), bg="#2c3e50", fg="white")
        title.pack(pady=20)
        
        subtitle = tk.Label(title_frame, text="Unified Data Processing & Analysis System", 
                           font=("Arial", 10), bg="#2c3e50", fg="#ecf0f1")
        subtitle.pack()
        
        # Main container
        main_frame = tk.Frame(self.root, padx=20, pady=20)
        main_frame.pack(fill="both", expand=True)
        
        # Step 1: Folder Selection
        step1 = ttk.LabelFrame(main_frame, text="Step 1: Configure Folders", padding=15)
        step1.pack(fill="x", pady=10)
        
        folder_frame = tk.Frame(step1)
        folder_frame.pack(fill="x")
        
        tk.Label(folder_frame, text="Input Folder:", width=20, anchor="w").grid(row=0, column=0, sticky="w", pady=5)
        self.input_folder_var = tk.StringVar(value=self.config.get_folder('input_folder'))
        tk.Entry(folder_frame, textvariable=self.input_folder_var, width=50, state='readonly').grid(row=0, column=1, padx=5)
        ttk.Button(folder_frame, text="Browse", command=self.select_input_folder).grid(row=0, column=2, padx=5)
        
        tk.Label(folder_frame, text="CSV Output:", width=20, anchor="w").grid(row=1, column=0, sticky="w", pady=5)
        self.csv_folder_var = tk.StringVar(value=self.config.get_folder('csv_output'))
        tk.Entry(folder_frame, textvariable=self.csv_folder_var, width=50, state='readonly').grid(row=1, column=1, padx=5)
        ttk.Button(folder_frame, text="Browse", command=self.select_csv_folder).grid(row=1, column=2, padx=5)
        
        tk.Label(folder_frame, text="Results Output:", width=20, anchor="w").grid(row=2, column=0, sticky="w", pady=5)
        self.output_folder_var = tk.StringVar(value=self.config.get_folder('results_output'))
        tk.Entry(folder_frame, textvariable=self.output_folder_var, width=50, state='readonly').grid(row=2, column=1, padx=5)
        ttk.Button(folder_frame, text="Browse", command=self.select_output_folder).grid(row=2, column=2, padx=5)
        
        # Step 2: Upload Files
        step2 = ttk.LabelFrame(main_frame, text="Step 2: Upload Excel Files", padding=15)
        step2.pack(fill="x", pady=10)
        
        upload_frame = tk.Frame(step2)
        upload_frame.pack(fill="x")
        
        tk.Label(upload_frame, text="Personal Info Files:", width=20, anchor="w").grid(row=0, column=0, sticky="w", pady=5)
        self.pi_label = tk.Label(upload_frame, text="No files selected", fg="gray", anchor="w")
        self.pi_label.grid(row=0, column=1, sticky="w", padx=5)
        ttk.Button(upload_frame, text="Upload PI Files", command=self.upload_pi_files).grid(row=0, column=2, padx=5)
        
        tk.Label(upload_frame, text="Engagement Files:", width=20, anchor="w").grid(row=1, column=0, sticky="w", pady=5)
        self.eng_label = tk.Label(upload_frame, text="No files selected", fg="gray", anchor="w")
        self.eng_label.grid(row=1, column=1, sticky="w", padx=5)
        ttk.Button(upload_frame, text="Upload Engagement Files", command=self.upload_engagement_files).grid(row=1, column=2, padx=5)
        
        # Eligibility Mode Selection
        tk.Label(upload_frame, text="Eligibility Mode:", width=20, anchor="w").grid(row=2, column=0, sticky="w", pady=5)
        self.eligibility_var = tk.StringVar(value='relaxed')
        eligibility_combo = ttk.Combobox(upload_frame, textvariable=self.eligibility_var, 
                                        values=['strict', 'moderate', 'relaxed', 'basic'],
                                        state='readonly', width=15)
        eligibility_combo.grid(row=2, column=1, sticky="w", padx=5)
        
        help_btn = ttk.Button(upload_frame, text="?", width=3, 
                            command=lambda: messagebox.showinfo("Eligibility Modes",
                                "strict: Requires 4 buckets\n"
                                "moderate: Requires 3 buckets\n"
                                "relaxed: Requires 2 buckets\n"
                                "basic: Requires 1 bucket\n\n"
                                "Doctors not meeting criteria go to Cluster 0"))
        help_btn.grid(row=2, column=2, padx=5)
        
        # Step 3: Convert & Analyze
        step3 = ttk.LabelFrame(main_frame, text="Step 3: Process & Analyze", padding=15)
        step3.pack(fill="x", pady=10)
        
        button_frame = tk.Frame(step3)
        button_frame.pack()
        
        self.convert_btn = ttk.Button(button_frame, text="1. Convert Excel → CSV", 
                                      command=self.convert_files, width=30)
        self.convert_btn.pack(side="left", padx=10, pady=10)
        
        self.analyze_btn = ttk.Button(button_frame, text="2. Run PSIms Analysis", 
                                     command=self.run_analysis, width=30, state='disabled')
        self.analyze_btn.pack(side="left", padx=10, pady=10)
        
        # Status Log
        status_frame = ttk.LabelFrame(main_frame, text="Status Log", padding=10)
        status_frame.pack(fill="both", expand=True, pady=10)
        
        self.status_text = scrolledtext.ScrolledText(status_frame, height=15, width=100, 
                                                     font=("Courier", 9), wrap=tk.WORD)
        self.status_text.pack(fill="both", expand=True)
        
        # Initial log message
        self.log("✓ PSIms System Ready")
        self.log("→ Please configure folders and upload files to begin")
    
    def log(self, message):
        """Add message to status log"""
        timestamp = datetime.now().strftime('%H:%M:%S')
        self.status_text.insert("end", f"[{timestamp}] {message}\n")
        self.status_text.see("end")
        self.root.update_idletasks()
    
    def select_input_folder(self):
        folder = filedialog.askdirectory(title="Select Input Folder")
        if folder:
            self.config.update_folder('input_folder', folder)
            self.input_folder_var.set(folder)
            self.log(f"✓ Input folder: {folder}")
    
    def select_csv_folder(self):
        folder = filedialog.askdirectory(title="Select CSV Output Folder")
        if folder:
            self.config.update_folder('csv_output', folder)
            self.csv_folder_var.set(folder)
            self.log(f"✓ CSV output: {folder}")
    
    def select_output_folder(self):
        folder = filedialog.askdirectory(title="Select Results Output Folder")
        if folder:
            self.config.update_folder('results_output', folder)
            self.output_folder_var.set(folder)
            self.log(f"✓ Results output: {folder}")
    
    def upload_pi_files(self):
        files = filedialog.askopenfilenames(
            title="Select Personal Info Files (Batch 1, Batch 2, etc.)",
            filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
        )
        if files:
            self.pi_files = list(files)
            self.pi_label.config(text=f"{len(files)} file(s) selected", fg="green")
            self.log(f"✓ Uploaded {len(files)} PI file(s):")
            for f in files:
                self.log(f"   - {os.path.basename(f)}")
    
    def upload_engagement_files(self):
        files = filedialog.askopenfilenames(
            title="Select Engagement Files (Jan'25, Feb'25, etc.)",
            filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
        )
        if files:
            self.engagement_files = list(files)
            self.eng_label.config(text=f"{len(files)} file(s) selected", fg="green")
            self.log(f"✓ Uploaded {len(files)} engagement file(s):")
            for f in files:
                self.log(f"   - {os.path.basename(f)}")
    
    def convert_files(self):
        """Convert Excel files to CSV"""
        if not self.pi_files and not self.engagement_files:
            messagebox.showerror("Error", "Please upload files first!")
            return
        
        csv_folder = self.config.get_folder('csv_output')
        if not csv_folder:
            messagebox.showerror("Error", "Please select CSV output folder!")
            return
        
        # Create folder if doesn't exist
        Path(csv_folder).mkdir(parents=True, exist_ok=True)
        
        try:
            self.log("\n" + "="*70)
            self.log("STARTING EXCEL TO CSV CONVERSION")
            self.log("="*70)
            
            converter = SmartConverter(csv_folder, log_callback=self.log)
            
            if self.pi_files:
                converter.combine_pi_batches(self.pi_files)
            
            if self.engagement_files:
                converter.convert_engagement_files(self.engagement_files)
            
            # Check for warnings
            warnings = converter.get_warnings()
            if warnings:
                warning_msg = "Fuzzy matching was used for the following:\n\n" + "\n".join(warnings)
                warning_msg += "\n\nPlease verify the data is correct."
                messagebox.showwarning("Fuzzy Matching Applied", warning_msg)
            
            self.log("="*70)
            self.log("✓ CONVERSION COMPLETE")
            self.log("="*70)
            
            # Enable analysis button
            self.analyze_btn.config(state='normal')
            
            messagebox.showinfo("Success", "Excel files converted to CSV successfully!\n\nYou can now run PSIms Analysis.")
            
        except Exception as e:
            self.log(f"\n❌ Conversion failed: {e}")
            messagebox.showerror("Error", f"Conversion failed:\n{str(e)}")
    
    def run_analysis(self):
        """Run PSIms analysis"""
        csv_folder = self.config.get_folder('csv_output')
        output_folder = self.config.get_folder('results_output')
        
        if not csv_folder or not output_folder:
            messagebox.showerror("Error", "Please configure all folders!")
            return
        
        # Check if CSV files exist
        if not os.path.exists(csv_folder) or not os.listdir(csv_folder):
            messagebox.showerror("Error", "No CSV files found! Please convert Excel files first.")
            return
        
        # Create output folder
        Path(output_folder).mkdir(parents=True, exist_ok=True)
        
        try:
            self.log("\n" + "="*70)
            self.log("STARTING PSIMS ANALYSIS")
            self.log("="*70)
            
            # Get selected eligibility mode
            eligibility_mode = self.eligibility_var.get()
            self.log(f"Eligibility Mode: {eligibility_mode}")
            
            engine = PSImsEngine(csv_folder, output_folder, log_callback=self.log, eligibility_mode=eligibility_mode)
            result_file = engine.run_full_analysis()
            
            if result_file:
                self.log("="*70)
                self.log("✓ ANALYSIS COMPLETE")
                self.log("="*70)
                
                # Show success with option to open folder
                msg = f"Analysis completed successfully!\n\nResults saved to:\n{result_file}\n\nWould you like to open the output folder?"
                if messagebox.askyesno("Success", msg):
                    os.startfile(output_folder)
            else:
                messagebox.showerror("Error", "Analysis failed. Check the log for details.")
        
        except Exception as e:
            self.log(f"\n❌ Analysis failed: {e}")
            import traceback
            self.log(traceback.format_exc())
            messagebox.showerror("Error", f"Analysis failed:\n{str(e)}")
    
    def run(self):
        """Start the UI"""
        self.root.mainloop()


# =====================================================
# MAIN EXECUTION
# =====================================================

def main():
    """Main entry point"""
    print("Starting PSIms Unified System...")
    app = PSImsUI()
    app.run()


if __name__ == "__main__":
    main()

Starting PSIms Unified System...
